In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline


DATA_ROOT = Path(
    r"C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG\ds004148"
)

FEATURE_FILE = (
    DATA_ROOT /
    "processed_windows" /
    "combined_eeg_features.csv"
)

features_df = pd.read_csv(
    FEATURE_FILE
)

print(
    "Shape:",
    features_df.shape
)

print(
    features_df.head()
)

Shape: (8735, 111)
  subject       session  window  Fp1_delta_power  Fp1_delta_logpower  \
0  sub-01  ses-session1       0     3.838935e-11          -23.983241   
1  sub-01  ses-session1       1     6.353824e-11          -23.479379   
2  sub-01  ses-session1       2     1.508287e-11          -24.917461   
3  sub-01  ses-session1       3     2.788458e-11          -24.302947   
4  sub-01  ses-session1       4     4.087792e-11          -23.920431   

   Fp1_theta_power  Fp1_theta_logpower  Fp1_alpha_power  Fp1_alpha_logpower  \
0     4.979447e-12          -26.025702     2.132238e-12          -26.873849   
1     5.749345e-12          -25.881935     2.178940e-12          -26.852183   
2     3.777661e-12          -26.301916     4.369069e-12          -26.156471   
3     3.778646e-12          -26.301655     1.914700e-12          -26.981460   
4     6.327972e-12          -25.786041     1.378515e-12          -27.310015   

   Fp1_beta_power  ...  PLV_Fp1_Fz_delta  PLV_Fp1_Fz_theta  PLV_Fp1_Fz_al

In [2]:
META_COLUMNS = [
    "subject",
    "session",
    "window"
]

FEATURE_COLUMNS = [
    col
    for col in features_df.columns
    if col not in META_COLUMNS
]

X = features_df[
    FEATURE_COLUMNS
]

# Replace inf with NaN
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

print(
    "NaN features:",
    X.isna().sum().sum()
)

# Median imputation
X = X.fillna(
    X.median()
)

features_df[
    FEATURE_COLUMNS
] = X

print(
    "Remaining NaN:",
    features_df[
        FEATURE_COLUMNS
    ].isna().sum().sum()
)

NaN features: 0
Remaining NaN: 0


In [3]:
ENROLL_SESSIONS = [
    "ses-session1",
    "ses-session2"
]

VERIFY_SESSION = "ses-session3"


enrollment_df = features_df[
    features_df["session"].isin(
        ENROLL_SESSIONS
    )
].copy()

verification_df = features_df[
    features_df["session"] ==
    VERIFY_SESSION
].copy()


print(
    "Enrollment:",
    enrollment_df.shape
)

print(
    "Verification:",
    verification_df.shape
)

Enrollment: (6100, 111)
Verification: (2635, 111)


In [4]:
svm_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "svm",
        SVC(
            kernel="rbf",
            probability=True,
            random_state=42
        )
    )
])


svm_model.fit(
    enrollment_df[FEATURE_COLUMNS],
    enrollment_df["subject"]
)

print("Model trained.")

c:\Users\Sanjeev\OneDrive\Desktop\Brain ID\Model\EEG-Authentication\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Model trained.


In [5]:
verification_probabilities = (
    svm_model.predict_proba(
        verification_df[
            FEATURE_COLUMNS
        ]
    )
)

classes = svm_model.classes_

trial_rows = []

for i, (_, row) in enumerate(
    verification_df.iterrows()
):

    true_subject = row["subject"]

    for class_idx, claimed_subject in enumerate(
        classes
    ):

        score = (
            verification_probabilities[
                i,
                class_idx
            ]
        )

        genuine = (
            claimed_subject ==
            true_subject
        )

        trial_rows.append({

            "true_subject":
                true_subject,

            "claimed_subject":
                claimed_subject,

            "window":
                row["window"],

            "score":
                score,

            "genuine":
                genuine
        })


trials_df = pd.DataFrame(
    trial_rows
)

print(
    trials_df.shape
)

(158100, 5)


In [6]:
def calculate_far_frr(
    trials,
    threshold
):

    accepted = (
        trials["score"] >= threshold
    )

    genuine = trials["genuine"]
    impostor = ~genuine

    false_rejects = (
        (~accepted) & genuine
    ).sum()

    genuine_count = genuine.sum()

    false_accepts = (
        accepted & impostor
    ).sum()

    impostor_count = impostor.sum()

    frr = (
        false_rejects / genuine_count
        if genuine_count > 0
        else np.nan
    )

    far = (
        false_accepts / impostor_count
        if impostor_count > 0
        else np.nan
    )

    tar = 1 - frr

    return far, frr, tar

In [7]:
thresholds = np.linspace(
    0,
    1,
    1001
)

results = []

for threshold in thresholds:

    far, frr, tar = calculate_far_frr(
        trials_df,
        threshold
    )

    results.append({
        "threshold": threshold,
        "FAR": far,
        "FRR": frr,
        "TAR": tar
    })


metrics_df = pd.DataFrame(
    results
)

metrics_df["difference"] = (
    metrics_df["FAR"] -
    metrics_df["FRR"]
).abs()


eer_row = metrics_df.loc[
    metrics_df["difference"].idxmin()
]

eer = (
    eer_row["FAR"] +
    eer_row["FRR"]
) / 2


print(
    "=============================="
)

print(
    f"EER      : {eer:.6f}"
)

print(
    f"EER %    : {eer * 100:.2f}%"
)

print(
    f"Threshold: {eer_row['threshold']:.4f}"
)

print(
    f"FAR      : {eer_row['FAR']:.6f}"
)

print(
    f"FRR      : {eer_row['FRR']:.6f}"
)

print(
    "=============================="
)

EER      : 0.255192
EER %    : 25.52%
Threshold: 0.0090
FAR      : 0.262188
FRR      : 0.248197
